# Đánh giá chất lượng mô hình RAG (Ragas Evaluation)

Notebook này thực hiện quá trình chấm điểm tự động hệ thống Hỏi-Đáp (RAG) bằng phương pháp **LLM-as-a-Judge**. Chúng ta sẽ đánh giá trên 2 tập dữ liệu:
1. **Easy Set**: Các câu hỏi ngắn gọn, trực diện, từ khóa rõ ràng.
2. **Hard Set**: Các câu hỏi dài, mang tính suy luận, lắt léo và đòi hỏi tổng hợp nhiều luồng thông tin.

**Các tiêu chí chấm điểm (Metrics):**
- `Context Recall`: Tài liệu truy xuất có chứa đáp án không?
- `Context Precision`: Tài liệu chứa đáp án có được xếp hạng Top 1 không?
- `Faithfulness`: Câu trả lời của AI có trung thực với tài liệu không (hay bị ảo giác - hallucination)?


## Import các thư viện & Cấu hình đường dẫn 

In [ ]:
import sys 
import json 
import nest_asyncio 
import pandas as pd 
from pathlib import Path 

nest_asyncio.apply()
sys.path.append(str(Path.cwd().parent))
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, context_precision, context_recall
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from backend.rag.retriever import retrieve_context
from backend.rag.generator import generate_answer
from backend.core.config import EMBEDDING_MODEL
print(" Đã import các thư viện thành công!")


 Đã import các thư viện thành công!
Đường dẫn hiện tại: None


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8196\1779684474.py:12: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_precision, context_recall
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8196\1779684474.py:12: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, context_precision, context_recall
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8196\1779684474.py:12: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import con

## Load dữ liệu từ JSON 

In [12]:
DATA_DIR = Path("../data/eval/datasets") 
# # Lấy đường dẫn và in ra màn hình
# my_path = str(Path.cwd().parent)
# print("Đường dẫn hiện tại là:", my_path)

with open(DATA_DIR / "easy_questions.json", "r", encoding="utf-8") as f: 
    easy_questions = json.load(f)

with open(DATA_DIR / "hard_questions.json", "r", encoding="utf-8") as f: 
    hard_questions = json.load(f)


print(f"Đọc dữ liệu từ JSON thành công !!")
df_easy = pd.DataFrame(easy_questions)
display(df_easy)


Đọc dữ liệu từ JSON thành công !!


,question,ground_truth
0,REIS là viết tắt của cụm từ gì?,REIS là viết tắt của Real-time Environmental I...
1,REIS tập trung vào lĩnh vực nào?,REIS là hệ thống giám sát và phân tích dữ liệu...
2,Hệ thống thu thập dữ liệu với tần suất bao nhiêu?,Hệ thống thu thập dữ liệu mỗi 15 phút.
3,REIS giám sát dữ liệu trên phạm vi bao nhiêu t...,REIS giám sát dữ liệu trên toàn bộ 63 tỉnh thà...
4,Chỉ số môi trường chính được theo dõi trong đồ...,Chỉ số chính được theo dõi là AQI (Air Quality...
5,Dual-AI Engine gồm những thành phần nào?,Dual-AI Engine gồm Anomaly Detection và Foreca...
6,Mô hình nào được sử dụng làm baseline phát hiệ...,Baseline phát hiện bất thường sử dụng phương p...
7,REIS sử dụng hệ quản trị cơ sở dữ liệu nào để ...,REIS sử dụng TimescaleDB để lưu trữ dữ liệu ch...
8,Redis được sử dụng với mục đích gì?,Redis được sử dụng để cache dữ liệu và giảm số...
9,AI Assistant sử dụng kỹ thuật nào để trả lời c...,AI Assistant sử dụng kỹ thuật Retrieval-Augmen...


## Chuẩn bị và xử lí Dataset cho bước đánh giá 

In [13]:
def prepare_ragas_dataset(question_list): 
    questions = []
    answers = []
    contexts = []
    ground_truths = []

    for item in question_list: 
        q = item["question"]
        print(f"Đang xử lý: {q}")
        retrieved_chunks = retrieve_context(q, k=3, use_hyde=True, use_reranker=True)
        ans = generate_answer(q, retrieved_chunks)
        ctx_texts = [chunk["text"] for chunk in retrieved_chunks]
        
        questions.append(q)
        answers.append(ans)
        contexts.append(ctx_texts)
        ground_truths.append(item["ground_truth"])
        
    data = {
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths,
    }
    return Dataset.from_dict(data)

# Chạy với bộ easy 
easy_dataset = prepare_ragas_dataset(easy_questions)
print("Hoàn tất chuẩn bị Dataset Easy!")

Đang xử lý: REIS là viết tắt của cụm từ gì?
Connecting to ChromaDB at D:\2025-2026 HKII\multimodel_e_learning\data\chroma_db
Loading embedding model: BAAI/bge-m3


d:\2025-2026 HKII\multimodel_e_learning\backend\db\vector_store.py:55: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  _vectorstore = Chroma(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Đang khởi tạo mô hình Gemini: gemini-3.1-flash-lite
Đang sinh câu trả lời giả định (HyDE)...
Sử dụng HyDE Document để search: REIS là viết tắt của cụm từ tiếng Anh "Real Estate Investment System", thường được sử dụng để chỉ cá...


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Loading Cross-Encoder Reranker (BAAI/bge-reranker-v2-m3)...
Đang chạy Cross-Encoder Reranker để chấm điểm và xếp hạng lại...
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: REIS tập trung vào lĩnh vực nào?
Đang sinh câu trả lời giả định (HyDE)...
Sử dụng HyDE Document để search: REIS (Real Estate Investment Strategy) tập trung chủ yếu vào lĩnh vực đầu tư và quản lý bất động sản...
Đang chạy Cross-Encoder Reranker để chấm điểm và xếp hạng lại...
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: Hệ thống thu thập dữ liệu với tần suất bao nhiêu?
Đang sinh câu trả lời giả định (HyDE)...
Sử dụng HyDE Document để search: Tần suất thu thập dữ liệu trong hệ thống E-Learning phụ thuộc vào cấu hình của từng module, thường d...
Đang chạy Cross-Encoder Reranker để chấm điểm và xếp hạng lại...
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: REIS giám sát dữ liệu trên phạm vi bao nhiêu tỉnh thành?
Đang sinh câu trả lời giả định (HyDE)...
Sử dụng HyDE 

## Chấm điểm RAGAS (trước khi dùng HyDE + Cross Encoder)

In [20]:
def prepare_ragas_dataset_no_advanced(question_list): 
    questions = []
    answers = []
    contexts = []
    ground_truths = []

    for item in question_list: 
        q = item["question"]
        print(f"Đang xử lý: {q}")
        retrieved_chunks = retrieve_context(q, k=3, use_hyde=False, use_reranker=False)
        ans = generate_answer(q, retrieved_chunks)
        ctx_texts = [chunk["text"] for chunk in retrieved_chunks]
        
        questions.append(q)
        answers.append(ans)
        contexts.append(ctx_texts)
        ground_truths.append(item["ground_truth"])
        
    data = {
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths,
    }
    return Dataset.from_dict(data)

# Chạy với bộ easy 
easy_dataset_no_advanced = prepare_ragas_dataset_no_advanced(easy_questions)
print("Hoàn tất chuẩn bị Dataset Easy!")

Đang xử lý: REIS là viết tắt của cụm từ gì?
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: REIS tập trung vào lĩnh vực nào?
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: Hệ thống thu thập dữ liệu với tần suất bao nhiêu?
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: REIS giám sát dữ liệu trên phạm vi bao nhiêu tỉnh thành?
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: Chỉ số môi trường chính được theo dõi trong đồ án là gì?
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: Dual-AI Engine gồm những thành phần nào?
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: Mô hình nào được sử dụng làm baseline phát hiện bất thường?
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: REIS sử dụng hệ quản trị cơ sở dữ liệu nào để lưu trữ dữ liệu chuỗi thời gian?
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: Redis được sử dụng với mục đích gì?
Đang gửi câu hỏi kèm ng

In [21]:
import os
from dotenv import load_dotenv
env_path = Path("../.env")
load_dotenv(dotenv_path=env_path)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
judge_llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.0 
)

judge_embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
print("Bắt đầu quá trình chấm điểm....")

result_no_advanced = evaluate(
    easy_dataset_no_advanced, 
    metrics = [
        faithfulness, 
        context_precision, 
        context_recall
    ], 
    llm = judge_llm, 
    embeddings = judge_embeddings
)
print(f"Điểm số bộ easy")
print(result_no_advanced)

# Tự động tạo thư mục ở root project (lùi ra 1 bước bằng ../)
os.makedirs("../reports/ragas", exist_ok=True)

# Lưu ra file ở root project
df_easy.to_csv("../reports/ragas/_no_advanced.csv", index=False, encoding='utf-8-sig')
print("✅ Đã lưu kết quả chi tiết vào ../reports/ragas/ragas_baseline_easy_no_advanced.csv")


Bắt đầu quá trình chấm điểm....


Evaluating:   0%|          | 0/45 [00:00<?, ?it/s]

Exception raised in Job[12]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqbysgh1fmdrh2k1w1md24ge` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99920, Requested 436. Please try again in 5m7.584s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[9]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqbysgh1fmdrh2k1w1md24ge` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99906, Requested 1413. Please try again in 18m59.616s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[13]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limi

Điểm số bộ easy
{'faithfulness': nan, 'context_precision': nan, 'context_recall': nan}
✅ Đã lưu kết quả chi tiết vào ../reports/ragas/ragas_baseline_easy_no_advanced.csv


## Chấm điểm RAGAS (sau khi dùng HyDE + Cross Encoder)

In [19]:
import os
from dotenv import load_dotenv
env_path = Path("../.env")
load_dotenv(dotenv_path=env_path)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
judge_llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.0 
)

judge_embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
print("Bắt đầu quá trình chấm điểm....")

result = evaluate(
    easy_dataset, 
    metrics = [
        faithfulness, 
        context_precision, 
        context_recall
    ], 
    llm = judge_llm, 
    embeddings = judge_embeddings
)
print(f"Điểm số bộ easy")
print(result)

# Tự động tạo thư mục ở root project (lùi ra 1 bước bằng ../)
os.makedirs("../reports/ragas", exist_ok=True)

# Lưu ra file ở root project
df_easy.to_csv("../reports/ragas/ragas_baseline_easy.csv", index=False, encoding='utf-8-sig')
print("✅ Đã lưu kết quả chi tiết vào ../reports/ragas/ragas_baseline_easy.csv")


Bắt đầu quá trình chấm điểm....


Evaluating:   0%|          | 0/45 [00:00<?, ?it/s]

Exception raised in Job[10]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqbysgh1fmdrh2k1w1md24ge` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99326, Requested 1413. Please try again in 10m38.496s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[13]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqbysgh1fmdrh2k1w1md24ge` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99318, Requested 1607. Please try again in 13m19.199999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[19]: RateLimitError(Error code: 429 - {'error': {'message': 

Điểm số bộ easy
{'faithfulness': 0.6667, 'context_precision': nan, 'context_recall': 1.0000}
✅ Đã lưu kết quả chi tiết vào ../reports/ragas/ragas_baseline_easy.csv
